Cell 1 — Imports

In [1]:
import time
import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import normalize

Cell 2 — Config (all embeddings + timing params)

In [2]:
emb_folder = "E:\\Experiments\\Similarity Serach\\Embeddings"
EMBEDDING_FILES = {
    "bert_finetuned": emb_folder + "\\bert_finetuned_embeddings.xlsx",
    "gemini": emb_folder + "\\Gemini_Embedding.xlsx",
    "qwen3_8b": emb_folder + "\\Qwen3_Embedding_8B.xlsx",
    "sbert": emb_folder + "\\SBERT_Embedding_2_classification.xlsx",
}

# number of sampled queries per embedding file
N_QUERIES = 200

# number of neighbors to retrieve
TOPK = 20

# IVF params
IVF_NLIST = 128
IVF_NPROBE = 10

# PQ params
PQ_NLIST = 128
PQ_NPROBE = 10
PQ_NBITS = 8

# HNSW params
HNSW_M = 32
HNSW_EF_SEARCH = 64

# output
OUT_XLSX = "index_latency_results.xlsx"

Cell 3 — Loader for all embedding files

In [3]:
def is_numeric_col_name(c):
    if isinstance(c, (int, np.integer)):
        return True
    s = str(c)
    return s.isdigit()

def load_embedding_xlsx(path):
    df = pd.read_excel(path, engine="openpyxl")

    # label column
    label_candidates = [c for c in df.columns if str(c).lower() in ("label", "y", "class")]
    if not label_candidates:
        raise ValueError(f"{path} -> label column not found")
    label_col = label_candidates[0]

    # id column
    id_candidates = [c for c in df.columns if str(c).lower() in ("filename", "file", "text_file", "id", "file_id")]
    if id_candidates:
        preferred = [c for c in id_candidates if str(c).lower() in ("filename", "file", "text_file")]
        id_col = preferred[0] if preferred else id_candidates[0]
    else:
        non_num = [c for c in df.columns if c != label_col and not is_numeric_col_name(c)]
        if not non_num:
            raise ValueError(f"{path} -> id column not found")
        id_col = non_num[0]

    # embedding columns
    emb_cols = [c for c in df.columns if c not in (label_col, id_col) and (
        is_numeric_col_name(c) or str(c).lower().startswith("e") or str(c).lower().startswith("emb")
    )]

    if not emb_cols:
        raise ValueError(f"{path} -> embedding columns not found")

    X = df[emb_cols].to_numpy(dtype=np.float32)
    X = normalize(X)
    ids = df[id_col].astype(str).to_numpy()

    return df, X, ids

Cell 4 — Query sampler

In [4]:
def sample_queries(X, n_queries=200, seed=42):
    rng = np.random.default_rng(seed)

    if n_queries > len(X):
        n_queries = len(X)

    q_idx = rng.choice(len(X), size=n_queries, replace=False)

    queries = X[q_idx]

    return queries

Cell 5 — Flat index latency measurement

In [5]:
def measure_flat_latency(X, queries, topk):

    N, d = X.shape

    index = faiss.IndexFlatIP(d)
    index.add(X)

    # warm-up
    for q in queries[:10]:
        index.search(q.reshape(1,-1), topk)

    t0 = time.time()

    for q in queries:
        index.search(q.reshape(1,-1), topk)

    avg_ms = (time.time() - t0) / len(queries) * 1000

    return avg_ms

Cell 6 — IVF index latency measurement

In [6]:
def measure_ivf_latency(X, queries, topk):

    N, d = X.shape

    quantizer = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFFlat(quantizer, d, IVF_NLIST, faiss.METRIC_INNER_PRODUCT)

    index.train(X)
    index.add(X)

    index.nprobe = IVF_NPROBE

    # warm-up
    for q in queries[:10]:
        index.search(q.reshape(1,-1), topk)

    t0 = time.time()

    for q in queries:
        index.search(q.reshape(1,-1), topk)

    avg_ms = (time.time() - t0) / len(queries) * 1000

    return avg_ms

Cell 7 — IVFPQ index latency measurement

In [7]:
def measure_ivfpq_latency(X, queries, topk):

    N, d = X.shape

    # number of subquantizers
    M = d // 8 if d % 8 == 0 else max(1, d // 8)

    quantizer = faiss.IndexFlatIP(d)
    index = faiss.IndexIVFPQ(quantizer, d, PQ_NLIST, M, PQ_NBITS)

    index.train(X)
    index.add(X)

    index.nprobe = PQ_NPROBE

    # warm-up
    for q in queries[:10]:
        index.search(q.reshape(1, -1), topk)

    t0 = time.time()

    for q in queries:
        index.search(q.reshape(1, -1), topk)

    avg_ms = (time.time() - t0) / len(queries) * 1000

    return avg_ms

Cell 8 — HNSW index latency measurement

In [8]:
def measure_hnsw_latency(X, queries, topk):

    N, d = X.shape

    index = faiss.IndexHNSWFlat(d, HNSW_M)

    index.hnsw.efSearch = HNSW_EF_SEARCH

    index.add(X)

    # warm-up
    for q in queries[:10]:
        index.search(q.reshape(1,-1), topk)

    t0 = time.time()

    for q in queries:
        index.search(q.reshape(1,-1), topk)

    avg_ms = (time.time() - t0) / len(queries) * 1000

    return avg_ms

Cell 9 — Run latency experiment for all embeddings

In [9]:
all_latency_rows = []

for emb_name, path in EMBEDDING_FILES.items():
    print(f"\n=== Measuring latency for: {emb_name} ===")

    df, X, ids = load_embedding_xlsx(path)
    queries = sample_queries(X, n_queries=N_QUERIES, seed=42)

    # Flat
    t_flat = measure_flat_latency(X, queries, TOPK)
    print(f"  Flat  : {t_flat:.4f} ms")

    # IVF
    t_ivf = measure_ivf_latency(X, queries, TOPK)
    print(f"  IVF   : {t_ivf:.4f} ms")

    # IVFPQ
    t_ivfpq = measure_ivfpq_latency(X, queries, TOPK)
    print(f"  IVFPQ : {t_ivfpq:.4f} ms")

    # HNSW
    t_hnsw = measure_hnsw_latency(X, queries, TOPK)
    print(f"  HNSW  : {t_hnsw:.4f} ms")

    all_latency_rows.extend([
        {
            "embedding": emb_name,
            "method": "Flat",
            "avg_query_time_ms": t_flat,
            "num_queries": len(queries),
            "topk": TOPK
        },
        {
            "embedding": emb_name,
            "method": "IVF",
            "avg_query_time_ms": t_ivf,
            "num_queries": len(queries),
            "topk": TOPK
        },
        {
            "embedding": emb_name,
            "method": "IVFPQ",
            "avg_query_time_ms": t_ivfpq,
            "num_queries": len(queries),
            "topk": TOPK
        },
        {
            "embedding": emb_name,
            "method": "HNSW",
            "avg_query_time_ms": t_hnsw,
            "num_queries": len(queries),
            "topk": TOPK
        },
    ])

latency_df = pd.DataFrame(all_latency_rows)
latency_df


=== Measuring latency for: bert_finetuned ===
  Flat  : 0.2332 ms
  IVF   : 0.0602 ms
  IVFPQ : 0.2667 ms
  HNSW  : 0.0476 ms

=== Measuring latency for: gemini ===
  Flat  : 0.2385 ms
  IVF   : 0.0486 ms
  IVFPQ : 0.4613 ms
  HNSW  : 0.0719 ms

=== Measuring latency for: qwen3_8b ===
  Flat  : 2.3497 ms
  IVF   : 1.2566 ms
  IVFPQ : 1.8369 ms
  HNSW  : 0.2956 ms

=== Measuring latency for: sbert ===
  Flat  : 0.1044 ms
  IVF   : 0.0300 ms
  IVFPQ : 0.3549 ms
  HNSW  : 0.0659 ms


,embedding,method,avg_query_time_ms,num_queries,topk
0,bert_finetuned,Flat,0.233171,200,20
1,bert_finetuned,IVF,0.060240,200,20
2,bert_finetuned,IVFPQ,0.266737,200,20
3,bert_finetuned,HNSW,0.047649,200,20
4,gemini,Flat,0.238532,200,20
5,gemini,IVF,0.048597,200,20
6,gemini,IVFPQ,0.461303,200,20
7,gemini,HNSW,0.071933,200,20
8,qwen3_8b,Flat,2.349710,200,20
9,qwen3_8b,IVF,1.256561,200,20


Cell 10 — Save latency results to Excel

In [ ]:
with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    latency_df.to_excel(writer, sheet_name="LatencyResults", index=False)

print("Saved:", OUT_XLSX)

✅ Saved: index_latency_results.xlsx
